# Gleich und Wechselstrombrücken

In [85]:
import uncertainties as unc
from uncertainties import ufloat
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sympy as smp

## Geräte Parameter

### 1.1 Berechnungen

Näherungsformel GLeichung 2.5. Widerstandsinkrement R_1 tauschen mit R_4 da $R_1R_4 = R_2R_3$ gilt



Abgleichbedingung

$$ 
\frac{R_1}{R_2} = \frac{R_3}{R_4}, \quad \frac{R_2}{R_1} = ratio \\
R_3 = R_4 \cdot ratio, \quad R_2 = \frac{R_1}{ratio}
$$

In [86]:
# Gegeben
UH = 3
# Zu Messen
deltaU = -341e-6
deltaR4 = 1
R1 = ufloat(3472,20) # Messergebnis +/- Garantiefehler im MB


In [87]:
def calcR(ratio):
    R4 = -(deltaR4 * UH) / (deltaU * (1+1/ratio)*(1+ratio))
    R3 = R4/ratio
    R2 = R1*ratio
    return {
        "Ratio": ratio,
        "R1": R1,
        "R2": R2,
        "R3": R3,
        "R4": R4
    }

data = [calcR(r) for r in [0.5, 1, 2]]
df1 = pd.DataFrame(data, columns=['Ratio', 'R1' ,'R2', 'R3', 'R4'])
print(df1.to_latex(index=False, float_format="%.2f"))
df1


\begin{tabular}{rllrr}
\toprule
Ratio & R1 & R2 & R3 & R4 \\
\midrule
0.50 & 3472+/-20 & 1736+/-10 & 3910.07 & 1955.03 \\
1.00 & 3472+/-20 & 3472+/-20 & 2199.41 & 2199.41 \\
2.00 & 3472+/-20 & (6.94+/-0.04)e+03 & 977.52 & 1955.03 \\
\bottomrule
\end{tabular}



,Ratio,R1,R2,R3,R4
0,0.5,3472+/-20,1736+/-10,3910.068426,1955.034213
1,1.0,3472+/-20,3472+/-20,2199.413490,2199.413490
2,2.0,3472+/-20,(6.94+/-0.04)e+03,977.517107,1955.034213


### 1.2 Berechnungen

R2 R3 Grob,
R4 fein,
R1 berechnen

In [88]:
def calcR(ratio, R2, R3, R4):
    R1 = R2*R3/R4
    return {
        "Ratio": ratio,
        "R1": R1,
        "R2": R2,
        "R3": R3,
        "R4": R4
    }

data = [ calcR(r,R2,R3,R4) for r,R2,R3,R4 in [
        (0.5,ufloat(2200,22),   ufloat(4700,47), ufloat(2969,29.69+0.025)),
        (0.5,ufloat(2200,22),   ufloat(2200,22), ufloat(1387,13.87+0.025)),
        (1,  ufloat(4700,47),   ufloat(2200,22), ufloat(2968,29.68+0.025)),
        (1,  ufloat(4700,47),   ufloat(1000,10), ufloat(1351,13.51+0.025)),
        (2,  ufloat(10000,100), ufloat(1000,10), ufloat(2905,29.05+0.025)),
        (2,  ufloat(10000,100), ufloat(470,4.7), ufloat(1360,13.6+0.025))
    ]]
df2 = pd.DataFrame(data, columns=['Ratio', 'R1' ,'R2', 'R3', 'R4'])
print(df2.to_latex(
    float_format="{:.2f}".format,
    formatters={
        "R1": "{:.2uS}".format,
        "R2": "{:.2uS}".format,
        "R3": "{:.2uS}".format,
        "R4": "{:.2uS}".format
        }
    ))
df2

\begin{tabular}{lrllll}
\toprule
 & Ratio & R1 & R2 & R3 & R4 \\
\midrule
0 & 0.50 & 3483(60) & 2200(22) & 4700(47) & 2969(30) \\
1 & 0.50 & 3490(60) & 2200(22) & 2200(22) & 1387(14) \\
2 & 1.00 & 3484(60) & 4700(47) & 2200(22) & 2968(30) \\
3 & 1.00 & 3479(60) & 4700(47) & 1000(10) & 1351(14) \\
4 & 2.00 & 3442(60) & 1.000(10)e+04 & 1000(10) & 2905(29) \\
5 & 2.00 & 3456(60) & 1.000(10)e+04 & 470.0(4.7) & 1360(14) \\
\bottomrule
\end{tabular}



,Ratio,R1,R2,R3,R4
0,0.5,(3.48+/-0.06)e+03,2200+/-22,(4.70+/-0.05)e+03,2969+/-30
1,0.5,(3.49+/-0.06)e+03,2200+/-22,2200+/-22,1387+/-14
2,1.0,(3.48+/-0.06)e+03,(4.70+/-0.05)e+03,2200+/-22,2968+/-30
3,1.0,(3.48+/-0.06)e+03,(4.70+/-0.05)e+03,1000+/-10,1351+/-14
4,2.0,(3.44+/-0.06)e+03,(1.000+/-0.010)e+04,1000+/-10,2905+/-29
5,2.0,(3.46+/-0.06)e+03,(1.000+/-0.010)e+04,470+/-5,1360+/-14


### 1.3 Empfindlichkeit

In [89]:
deltaR1 = 10.1 # 5...10 Ohm
U0min = 170e-6 # Kleinst mögliche spannungsänderung -> Auflösung MB


#### Messung $U_0$

Diese Annäherung gilt für sehr kleine Abweichungen
$$
E = \frac{\partial U_0}{\partial\frac{\Delta R_1}{R_1}} \approx \frac{U_0}{\frac{\Delta R_1}{R_1}}
$$

absolut Kleinste Widerstandsänderung, die am Messgerät erkannt wird

$$
\Delta R_{1,\min} = \frac{\Delta R_1\cdot U_{0,\min}}{U_0}
$$

relativ Kleinste Widerstandsänderung, die am Messgerät erkannt wird

$$
\frac{\Delta R_{1,min}}{R_1} = \frac{\Delta R_1\cdot U_{0,\min}}{U_0\cdot R_1}
$$

In [90]:
# Zu Messen
U0 = [-2.2e-3, 7.5e-3, -1.6e-3]

# Berechnung
relR1 = deltaR1/df2["R1"][[0,2,4]]
E = U0/relR1
deltaR1min = [deltaR1*U0min/u for u in U0]
relDeltaR1min = deltaR1min/df2["R1"][[0,2,4]]

data = {
    "E": E,
    "deltaR1min": deltaR1min,
    "relDeltaR1min": relDeltaR1min
}

df3 = pd.DataFrame(data, columns=['E', 'deltaR1min', 'relDeltaR1min'])
print(df3.to_latex(
    float_format="{:.2f}".format,
    formatters={
        "E": "{:.2uS}".format,
        "deltaR1min": "{:.4f}".format,
        "relDeltaR1min": "{:.2uS}".format,
        }
    ))
df3

\begin{tabular}{llrl}
\toprule
 & E & deltaR1min & relDeltaR1min \\
\midrule
0 & -0.759(13) & -0.7805 & -0.0002241(39) \\
2 & 2.587(45) & 0.2289 & 6.57(11)e-05 \\
4 & -0.5453(94) & -1.0731 & -0.0003117(54) \\
\bottomrule
\end{tabular}



,E,deltaR1min,relDeltaR1min
0,-0.759+/-0.013,-0.780455,-0.000224+/-0.000004
2,2.59+/-0.04,0.228933,(6.57+/-0.11)e-05
4,-0.545+/-0.009,-1.073125,-0.000312+/-0.000005
